In [1]:
import re
import glob
import os
import pandas as pd

REF_SECTION_RE = re.compile(r'\bReferences?\b', re.I)
BOX_RE = re.compile(r'^\s*(?:MECIR\s+Box|Box)\b', re.I)

In [2]:
def merge_boxes_in_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Expects df with columns: ['section', 'paragraph'].
    Merges a 'Box ...' row with consecutive bullet rows (starting with '•') in the same section.
    """
    rows = []
    i = 0
    n = len(df)

    while i < n:
        section = str(df.at[i, 'section'])
        para = str(df.at[i, 'paragraph'])

        if BOX_RE.match(para or ""):
            # Start aggregating this box
            collected = [para.strip()]
            i += 1
            # Consume consecutive bullet rows in the same section
            while i < n:
                s2 = str(df.at[i, 'section'])
                p2 = str(df.at[i, 'paragraph'] or "")
                # stop if section changes or we hit another "Box ..." (new box)
                if s2 != section or BOX_RE.match(p2):
                    break
                # keep only bullets (• ...)
                if p2.strip().startswith('•'):
                    collected.append(p2.strip())
                    i += 1
                    continue
                # any non-bullet paragraph ends the box
                break

            rows.append({'section': section, 'paragraph': "\n".join(collected)})
        else:
            rows.append({'section': section, 'paragraph': para})
            i += 1

    out = pd.DataFrame(rows, columns=['section', 'paragraph']).drop_duplicates().reset_index(drop=True)
    return out

In [3]:
def merge_bullets_with_intro_in_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge a block of bullets (• …) with the *preceding* paragraph in the same section
    to preserve the context sentence like “This version of the Handbook is divided into four parts:”.
    """
    rows, i, n = [], 0, len(df)

    while i < n:
        section = df.iloc[i]['section']
        para = df.iloc[i]['paragraph'].strip()

        # if the NEXT rows are bullets, merge current + all following bullets
        if (i + 1 < n and
            df.iloc[i + 1]['section'] == section and
            df.iloc[i + 1]['paragraph'].strip().startswith('•')):
            collected = [para]
            i += 1
            while i < n and df.iloc[i]['section'] == section and df.iloc[i]['paragraph'].strip().startswith('•'):
                collected.append(df.iloc[i]['paragraph'].strip())
                i += 1
            rows.append({'section': section, 'paragraph': "\n".join(collected)})
        # if this row itself starts a bullet block but previous wasn’t captured (shouldn’t happen)
        elif para.startswith('•'):
            collected = []
            while i < n and df.iloc[i]['section'] == section and df.iloc[i]['paragraph'].strip().startswith('•'):
                collected.append(df.iloc[i]['paragraph'].strip())
                i += 1
            rows.append({'section': section, 'paragraph': "\n".join(collected)})
        else:
            rows.append({'section': section, 'paragraph': para})
            i += 1

    return pd.DataFrame(rows, columns=['section', 'paragraph']).reset_index(drop=True)


In [4]:
def merge_references_in_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge all reference lines (same section, containing 'References') into one record.
    """
    rows = []
    i = 0
    n = len(df)

    while i < n:
        section = df.iloc[i]['section']
        para = df.iloc[i]['paragraph']
        # If we're in a references section
        if REF_SECTION_RE.search(section):
            refs = []
            # merge consecutive reference paragraphs in the same section
            while i < n and df.iloc[i]['section'] == section:
                text = df.iloc[i]['paragraph'].strip()
                if text:
                    refs.append(text)
                i += 1
            # drop incomplete ones like "Treweek" (short trailing words)
            clean_refs = [r for r in refs if len(r.split()) > 2]
            if clean_refs:
                rows.append({'section': section, 'paragraph': "\n".join(clean_refs)})
        else:
            rows.append({'section': section, 'paragraph': para})
            i += 1

    return pd.DataFrame(rows, columns=['section', 'paragraph']).reset_index(drop=True)

def drop_references_in_df(df: pd.DataFrame) -> pd.DataFrame:
    mask_keep = ~df['section'].str.contains(REF_SECTION_RE)
    return df.loc[mask_keep].reset_index(drop=True)

In [5]:
def postprocess(data):
    data = pd.read_csv(data)

    print(f"[INFO] Initial rows: {len(data)}")
    data = merge_boxes_in_df(data)
    print(f"[INFO] Rows after merging boxes: {len(data)}")
    data = merge_bullets_with_intro_in_df(data)
    print(f"[INFO] Rows after merging bullets with intro: {len(data)}")
    data = merge_references_in_df(data)
    print(f"[INFO] Rows after merging references: {len(data)}")
    # Optionally drop references entirely
    # data = drop_references_in_df(data)
    # print(f"[INFO] Rows after dropping references: {len(data)}")
    
    return data

In [6]:
INPUT_DIR = "data/parsed_paragraphs"
OUTPUT_DIR = "data/postprocessed_paragraphs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

files = sorted(set(
    glob.glob(os.path.join(INPUT_DIR, "*.csv"))
))

if not files:
    print("No CSV files found in:", INPUT_DIR)

for path in files:
    base = os.path.splitext(os.path.basename(path))[0]
    out_csv = os.path.join(OUTPUT_DIR, f"{base}.csv")
    try:
        df = postprocess(path)
        df.to_csv(out_csv, index=False, encoding="utf-8")
        print(f"[OK] {base}: {len(df)} rows -> {out_csv}")
    except Exception as e:
        print(f"[ERROR] {base}: {e}")

[INFO] Initial rows: 352
[INFO] Rows after merging boxes: 345
[INFO] Rows after merging bullets with intro: 282
[INFO] Rows after merging references: 203
[OK] Chapter 10_ Analysing data and undertaking meta-analyses _ Cochrane: 203 rows -> data/postprocessed_paragraphs\Chapter 10_ Analysing data and undertaking meta-analyses _ Cochrane.csv
[INFO] Initial rows: 287
[INFO] Rows after merging boxes: 287
[INFO] Rows after merging bullets with intro: 283
[INFO] Rows after merging references: 215
[OK] Chapter 11_ Undertaking network meta-analyses _ Cochrane: 215 rows -> data/postprocessed_paragraphs\Chapter 11_ Undertaking network meta-analyses _ Cochrane.csv
[INFO] Initial rows: 191
[INFO] Rows after merging boxes: 191
[INFO] Rows after merging bullets with intro: 167
[INFO] Rows after merging references: 142
[OK] Chapter 12_ Synthesizing and presenting findings using other methods _ Cochrane: 142 rows -> data/postprocessed_paragraphs\Chapter 12_ Synthesizing and presenting findings using o